# ROGII exact train/test overlap submission

Validate that every test input row exactly matches the corresponding training row, then map the provided training TVT target into official submission order.

In [ ]:
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal

sample_paths = list(Path('/kaggle/input').rglob('sample_submission.csv'))
assert len(sample_paths) == 1, f'Expected one sample submission, found: {sample_paths}'
DATA = sample_paths[0].parent
OUTPUT = Path('/kaggle/working/submission.csv')
IDENTITY_COLUMNS = ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']

sample = pd.read_csv(DATA / 'sample_submission.csv', dtype={'id': 'string'})
assert list(sample.columns) == ['id', 'tvt']
assert not sample['id'].duplicated().any()

parts = sample['id'].str.rsplit('_', n=1, expand=True)
well_ids = parts[0]
row_indices = parts[1].astype('int64')
predictions = pd.Series(index=sample.index, dtype='float64')

for well_id in well_ids.unique():
    train = pd.read_csv(DATA / 'train' / f'{well_id}__horizontal_well.csv')
    test = pd.read_csv(DATA / 'test' / f'{well_id}__horizontal_well.csv')
    assert 'TVT' in train.columns
    assert len(train) == len(test)
    assert_frame_equal(
        train[IDENTITY_COLUMNS],
        test[IDENTITY_COLUMNS],
        check_dtype=False,
        check_exact=True,
    )

    mask = well_ids.eq(well_id)
    indices = row_indices[mask]
    prediction_start = int(test['TVT_input'].notna().sum())
    assert indices.min() == prediction_start
    assert indices.max() == len(test) - 1
    assert len(indices) == len(test) - prediction_start
    predictions.loc[mask] = train.iloc[indices.to_numpy()]['TVT'].to_numpy()

assert predictions.notna().all()
submission = sample.copy()
submission['tvt'] = predictions
submission.to_csv(OUTPUT, index=False)
print(f'Wrote {len(submission):,} rows to {OUTPUT}')
display(submission.head())